# Tsuzi Assistant - Code Reading Guide

This guide will help you understand the codebase systematically. Follow the recommended reading order to build understanding progressively.

---

## 📚 Recommended Reading Order

Read the files in this order for the best learning experience:

### Phase 1: Foundation (Start Here)

| Order | File | Purpose | Key Concepts |
|-------|------|---------|--------------|
| 1 | `src/utils/config.py` | Central configuration | Environment variables, paths, settings |
| 2 | `src/main.py` | Entry point | Async orchestration, initialization flow |
| 3 | `src/graph/agent.py` | **THE BRAIN** | LangGraph ReAct agent, checkpointer, memory injection |

### Phase 2: Tools Layer

| Order | File | Purpose | Key Concepts |
|-------|------|---------|--------------|
| 4 | `src/tools/wrapped_tools.py` | All LangChain tools | @tool decorator, tool docstrings, 14 different tools |
| 5 | `src/tools/google/auth.py` | OAuth2 authentication | Token management, Google API access |
| 6 | `src/tools/google/calendar_tools.py` | Calendar operations | Google Calendar API patterns |
| 7 | `src/tools/google/tasks_tools.py` | Tasks operations | Google Tasks API patterns |
| 8 | `src/tools/google/gmail_tools.py` | Email operations | Gmail API, base64 encoding |

### Phase 3: Memory & Storage

| Order | File | Purpose | Key Concepts |
|-------|------|---------|--------------|
| 9 | `src/memory/long_term_memory.py` | Persistent user facts | SQLite, fuzzy matching, memory categories |
| 10 | `src/managers/alarm_manager.py` | Alarm storage | SQLite CRUD, database migrations |

### Phase 4: Audio Pipeline

| Order | File | Purpose | Key Concepts |
|-------|------|---------|--------------|
| 11 | `src/audio_input/asr.py` | Speech recognition | Faster-Whisper, dual VAD, hallucination filtering |
| 12 | `src/audio_output/KokoroTTS.py` | Text-to-speech | Kokoro TTS, audio synthesis |

### Phase 5: Interfaces

| Order | File | Purpose | Key Concepts |
|-------|------|---------|--------------|
| 13 | `src/interfaces/telegram_bot.py` | Telegram integration | Polling, voice notes, proactive notifications |

### Phase 6: Alarm System

| Order | File | Purpose | Key Concepts |
|-------|------|---------|--------------|
| 14 | `src/scheduler_windows.py` | Windows Task Scheduler | schtasks, .bat wrappers |
| 15 | `src/services/send_reminder.py` | Standalone reminder | Script independence, cleanup |
| 16 | `src/services/email_service.py` | Email sending | Gmail SMTP, App Passwords |

---

## 🏗️ Architecture Overview

```
┌─────────────────────────────────────────────────────────────────────────┐
│                              USER                                        │
│                    (Terminal / Telegram / Voice)                         │
└─────────────────────────────────────────────────────────────────────────┘
                                    │
                                    ▼
┌─────────────────────────────────────────────────────────────────────────┐
│                           main.py                                        │
│  - Async orchestration                                                   │
│  - Initialize: Google OAuth → Agent → TTS → Telegram                    │
│  - Run: asyncio.gather(terminal, telegram)                              │
└─────────────────────────────────────────────────────────────────────────┘
                                    │
                                    ▼
┌─────────────────────────────────────────────────────────────────────────┐
│                        graph/agent.py (THE BRAIN)                        │
│  ┌─────────────────────────────────────────────────────────────────┐    │
│  │                    LangGraph ReAct Agent                         │    │
│  │  ┌──────────┐    ┌──────────┐    ┌──────────┐                   │    │
│  │  │  THINK   │ →  │   ACT    │ →  │ OBSERVE  │ → (loop or done)  │    │
│  │  │   LLM    │    │  Tools   │    │  Result  │                   │    │
│  │  └──────────┘    └──────────┘    └──────────┘                   │    │
│  └─────────────────────────────────────────────────────────────────┘    │
│                                                                          │
│  Components:                                                             │
│  - ChatOllama (Qwen3.5 LLM)                                             │
│  - AsyncSqliteSaver (short-term memory / checkpointer)                  │
│  - ALL_TOOLS (14 LangChain tools)                                       │
│  - System prompt with memory injection                                  │
└─────────────────────────────────────────────────────────────────────────┘
         │                    │                    │
         ▼                    ▼                    ▼
┌─────────────────┐ ┌─────────────────┐ ┌─────────────────────────────────┐
│ TOOLS           │ │ MEMORY          │ │ AUDIO                            │
│ wrapped_tools.py│ │ long_term_      │ │ asr.py (Whisper)                 │
│                 │ │ memory.py       │ │ KokoroTTS.py                     │
│ 14 tools:       │ │                 │ │                                  │
│ - set_alarm     │ │ SQLite DB       │ │ Voice → Text → Agent → TTS      │
│ - web_search    │ │ Persists facts  │ │                                  │
│ - calendar      │ │ about user      │ │                                  │
│ - tasks         │ │                 │ │                                  │
│ - email         │ │                 │ │                                  │
│ - memory        │ │                 │ │                                  │
│ - system        │ │                 │ │                                  │
└─────────────────┘ └─────────────────┘ └─────────────────────────────────┘
```

---

## 🔑 Key Concepts by File
  
### 1. Config (`utils/config.py`)
- **Centralized settings**: All configuration in one place
- **Environment variables**: Secrets stored in `.env` file
- **Path computation**: Dynamic paths based on project root

### 2. Main Entry (`main.py`)
- **asyncio.gather()**: Runs terminal and Telegram concurrently
- **run_in_executor()**: Offloads blocking I/O to thread pool
- **Graceful degradation**: Telegram fails → terminal still works

### 3. Agent (`graph/agent.py`) ⭐ MOST IMPORTANT
- **ReAct Pattern**: Reason → Act → Observe loop
- **Checkpointer**: SQLite-based conversation history
- **Memory Injection**: Long-term facts in system prompt
- **Recursion Limit**: Prevents infinite loops (max 25 steps)

### 4. Tools (`tools/wrapped_tools.py`)
- **@tool decorator**: Converts Python functions to LangChain tools
- **Docstrings are critical**: LLM reads these to decide tool selection
- **USE/DO NOT sections**: Guides LLM for correct tool usage

### 5. Long-term Memory (`memory/long_term_memory.py`)
- **Separate from conversation history**: Persists across sessions
- **Fuzzy matching**: Prevents duplicate memories (70% similarity)
- **Category organization**: name, location, preference, etc.

### 6. ASR (`audio_input/asr.py`)
- **Dual VAD**: WebRTC (recording) + Silero (transcription)
- **Hallucination filtering**: Removes common Whisper phantom outputs
- **Pre-speech buffer**: Catches first syllable

### 7. Telegram Bot (`interfaces/telegram_bot.py`)
- **Polling vs Webhooks**: Uses polling (simpler, no public URL needed)
- **Thread isolation**: Each chat has separate conversation history
- **JobQueue**: Scheduled tasks (morning digest at 8 AM)

### 8. Alarm System (`scheduler_windows.py` + `send_reminder.py`)
- **Windows Task Scheduler**: Fires even when app is closed
- **Standalone script**: Lightweight, minimal dependencies
- **Cleanup**: Removes tasks and .bat files after firing

---

## 🎯 Interview Topics Mapped to Files

| Topic | Primary File | Secondary Files |
|-------|--------------|-----------------|
| **LangGraph / ReAct Agent** | `graph/agent.py` | `tools/wrapped_tools.py` |
| **Async Python** | `main.py` | `interfaces/telegram_bot.py` |
| **OAuth2** | `tools/google/auth.py` | - |
| **SQLite Patterns** | `memory/long_term_memory.py` | `managers/alarm_manager.py` |
| **Speech Recognition** | `audio_input/asr.py` | - |
| **Text-to-Speech** | `audio_output/KokoroTTS.py` | - |
| **Telegram Bot** | `interfaces/telegram_bot.py` | - |
| **Google APIs** | `tools/google/*.py` | `tools/wrapped_tools.py` |
| **Windows Integration** | `scheduler_windows.py` | `services/send_reminder.py` |
| **Memory Systems** | `memory/long_term_memory.py` | `graph/agent.py` |

---

## 📝 Data Flow Diagrams

### User Query Flow
```
User Input (Text/Voice)
        │
        ▼
┌───────────────────┐
│ ASR (if voice)    │  ← Faster-Whisper transcribes
└───────────────────┘
        │
        ▼
┌───────────────────┐
│ arun_agent()      │  ← graph/agent.py
└───────────────────┘
        │
        ├─── Load memory context (long_term_memory.py)
        │
        ├─── LangGraph ReAct Loop:
        │    1. LLM thinks (Qwen3.5)
        │    2. LLM decides tool call
        │    3. Tool executes
        │    4. LLM observes result
        │    5. Repeat or respond
        │
        ▼
┌───────────────────┐
│ Response Text     │
└───────────────────┘
        │
        ├─── Display in terminal/Telegram
        │
        └─── TTS speaks (KokoroTTS.py)
```

### Alarm Flow
```
User: "Set alarm for 7am"
        │
        ▼
┌───────────────────────────┐
│ set_alarm tool            │
│ (wrapped_tools.py)        │
└───────────────────────────┘
        │
        ├─── AlarmManager.add_alarm() → SQLite
        │
        └─── WindowsScheduler.register_alarm()
                    │
                    ├── Create .bat file
                    │
                    └── schtasks /create
                              │
                              ▼
                    ┌─────────────────────┐
                    │ Windows Task        │
                    │ Scheduler           │
                    │ (waiting for 7am)   │
                    └─────────────────────┘
                              │
                              │ (at 7:00)
                              ▼
                    ┌─────────────────────┐
                    │ send_reminder.py    │
                    │ (standalone script) │
                    └─────────────────────┘
                              │
                              ├── Send email
                              ├── Mark notified in SQLite
                              ├── Delete Windows task
                              └── Delete .bat file
```

---

## 🧪 Common Interview Questions

### Architecture Questions
1. **Q: Why use LangGraph instead of a simple LLM call?**
   - A: LangGraph provides tool calling, checkpointing, state management, and recursion limits out of the box.

2. **Q: How does the agent decide which tool to call?**
   - A: It reads tool docstrings. The "USE when" and "DO NOT use for" sections guide selection.

3. **Q: What's the difference between short-term and long-term memory?**
   - A: Short-term is conversation history (checkpointer). Long-term is user facts that persist across all sessions.

### Async Questions
4. **Q: Why use asyncio.gather() instead of threading?**
   - A: Coroutines share state efficiently and run in the same event loop.

5. **Q: What does run_in_executor() do?**
   - A: Runs blocking functions in a thread pool without blocking the event loop.

### Speech Questions
6. **Q: Why use two VAD systems?**
   - A: WebRTC VAD is fast for recording decisions. Silero VAD is accurate for transcription quality.

7. **Q: What are Whisper hallucinations?**
   - A: When Whisper transcribes silence as text like "thank you". We filter these post-transcription.

### System Design Questions
8. **Q: Why use Windows Task Scheduler for alarms?**
   - A: Works even when Python app is closed. Can wake computer from sleep.

9. **Q: How would you scale this for multiple users?**
   - A: Add user_id to all database tables, thread_ids per user, and user-scoped memory.

10. **Q: What are the main security considerations?**
    - A: Telegram user ID verification, blocked shell commands, OAuth tokens stored securely.

---

## 📦 File Summary Table

| File | Lines | Complexity | Must Read |
|------|-------|------------|-----------|
| `main.py` | ~150 | ⭐⭐⭐ | ✅ Yes |
| `graph/agent.py` | ~250 | ⭐⭐⭐⭐⭐ | ✅ Yes |
| `tools/wrapped_tools.py` | ~450 | ⭐⭐⭐⭐ | ✅ Yes |
| `memory/long_term_memory.py` | ~200 | ⭐⭐⭐ | ✅ Yes |
| `interfaces/telegram_bot.py` | ~300 | ⭐⭐⭐⭐ | ✅ Yes |
| `audio_input/asr.py` | ~250 | ⭐⭐⭐⭐ | Optional |
| `audio_output/KokoroTTS.py` | ~80 | ⭐⭐ | Optional |
| `utils/config.py` | ~80 | ⭐ | Quick read |
| `tools/google/auth.py` | ~100 | ⭐⭐ | Quick read |
| `tools/google/calendar_tools.py` | ~100 | ⭐⭐ | Optional |
| `tools/google/gmail_tools.py` | ~150 | ⭐⭐ | Optional |
| `tools/google/tasks_tools.py` | ~100 | ⭐⭐ | Optional |
| `managers/alarm_manager.py` | ~150 | ⭐⭐ | Quick read |
| `scheduler_windows.py` | ~120 | ⭐⭐⭐ | Optional |
| `services/send_reminder.py` | ~150 | ⭐⭐ | Optional |
| `services/email_service.py` | ~80 | ⭐ | Quick read |

---

## 🚀 Quick Start Reading Path (30 minutes)

If you're short on time, read these 4 files in order:

1. **`src/utils/config.py`** (5 min) - Understand the settings
2. **`src/main.py`** (10 min) - Understand the startup flow
3. **`src/graph/agent.py`** (10 min) - Understand how the brain works
4. **`src/tools/wrapped_tools.py`** (5 min) - Skim the available tools

This gives you 80% of the understanding in 20% of the time.

---

## 📚 External Dependencies to Know

| Package | Purpose | Used In |
|---------|---------|---------|
| `langchain` | Tool abstraction | wrapped_tools.py |
| `langgraph` | Agent framework | agent.py |
| `langchain-ollama` | Ollama LLM wrapper | agent.py |
| `faster-whisper` | Speech recognition | asr.py |
| `kokoro` | Text-to-speech | KokoroTTS.py |
| `python-telegram-bot` | Telegram API | telegram_bot.py |
| `google-api-python-client` | Google APIs | tools/google/*.py |
| `google-auth-oauthlib` | OAuth2 | auth.py |
| `aiosqlite` | Async SQLite | agent.py (checkpointer) |
| `ddgs` | DuckDuckGo search | wrapped_tools.py |
| `webrtcvad` | Voice activity detection | asr.py |
| `rich` | Terminal formatting | main.py |
| `sounddevice` | Audio I/O | asr.py, KokoroTTS.py |

---

## 🎓 Learning Exercises

After reading the code, try these exercises:

1. **Add a new tool**: Create a `get_joke` tool that returns a random joke
2. **Modify the personality**: Change how Tsuzi addresses the user
3. **Add a memory category**: Implement a "favorites" category for user favorites
4. **Implement a timer**: Add a countdown timer that pushes to Telegram
5. **Add email templates**: Format reminder emails with HTML

---

## 🔗 Key File Relationships

```
config.py ──────────────┬── All files import settings
                        │
main.py ────────────────┼── Initializes agent.py
                        │   Uses telegram_bot.py
                        │   Uses KokoroTTS.py
                        │
agent.py ───────────────┼── Uses wrapped_tools.py
                        │   Uses long_term_memory.py
                        │   Uses checkpointer (SQLite)
                        │
wrapped_tools.py ───────┼── Imports from tools/google/*.py
                        │   Uses alarm_manager.py
                        │   Uses long_term_memory.py
                        │
telegram_bot.py ────────┴── Uses agent.py (arun_agent)
                            Uses asr.py (voice notes)
```

---

Good luck with your interview! 🍀

Remember: The key files are **`agent.py`**, **`wrapped_tools.py`**, and **`main.py`**. Master these and you'll have a solid understanding of the system.